# Domain shift / degradation trend analysis

Questa analisi non usa il modello ST-GNN e non richiede training. Misura se la produzione reale degli impianti cala nel tempo rispetto a una baseline meteo/PVGIS.

- `pr_pvgis = ENERGIA reale / (kWp * PVGIS_POA)`: performance normalizzata rispetto a quanto ci si aspetta dato l'irraggiamento.
- `pr_month_z`: versione destagionalizzata mese per mese, cioe' confronto del PR giornaliero con media e deviazione standard del relativo mese.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs' / 'domain_shift_trend'
OUT

## Run dello script

Esegui questa cella per rigenerare CSV, summary e grafici. Se hai gia' generato gli output, puoi saltarla e caricare direttamente le celle successive.

In [ ]:
cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'analyze_domain_shift_trend.py'),
    '--out-dir', str(OUT),
    '--kwp-mode', 'real-only',
    '--plausible-pr-min', '0.2',
    '--plausible-pr-max', '2.0',
]
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT) + os.pathsep + env.get('PYTHONPATH', '')

print(' '.join(cmd))
res = subprocess.run(cmd, cwd=ROOT, env=env, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Script failed with exit code {res.returncode}')

## Summary numerico

In [ ]:
with open(OUT / 'summary.json', encoding='utf-8') as f:
    summary = json.load(f)

summary

In [ ]:
fleet_trends = pd.read_csv(OUT / 'fleet_trend_summary.csv')
plant_trends = pd.read_csv(OUT / 'plant_trend_summary.csv')
fleet_monthly = pd.read_csv(OUT / 'fleet_monthly_performance.csv', parse_dates=['date'])
plant_monthly = pd.read_csv(OUT / 'plant_monthly_performance.csv', parse_dates=['date'])

fleet_trends

## Grafici generati dallo script

In [ ]:
for name in ['fleet_pvgis_pr_trend.png', 'top_decreasing_plants_pvgis_pr.png']:
    path = OUT / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Missing: {path}')

## Impianti con trend decrescente piu' forte

In [ ]:
cols = [
    'metric', 'plant', 'plant_id', 'n_points', 'mean_value',
    'slope_per_year', 'relative_change_pct_per_year', 'p_value',
    'kendall_tau', 'decreasing'
]

pr_trends = plant_trends[plant_trends['metric'] == 'pr_pvgis_monthly'].copy()
pr_trends.sort_values('slope_per_year')[cols].head(20)

## Andamento fleet mensile

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fleet_monthly['date'], fleet_monthly['weighted_pr_pvgis'], 'o-', label='weighted PR PVGIS')
ax.plot(fleet_monthly['date'], fleet_monthly['median_pr_pvgis'], 's--', label='median plant PR')
ax.axhline(fleet_monthly['weighted_pr_pvgis'].mean(), color='0.3', linewidth=1, alpha=0.6)
ax.set_xlabel('Mese')
ax.set_ylabel('actual / (kWp * PVGIS POA)')
ax.set_title('Performance normalizzata PVGIS - flotta')
ax.legend()
plt.tight_layout()

## Lettura per la tesi

Un trend negativo di `pr_pvgis` indica che, a parita' di riferimento meteo/PVGIS e capacita' stimata o reale, la produzione cala nel tempo. Questo e' il segnale piu' vicino a soiling, degrado sistemico o domain shift temporale della flotta.

`pr_month_z` serve invece a ridurre la stagionalita': ogni giorno viene confrontato con la distribuzione del suo mese. Con un solo anno il risultato va letto come diagnostica esplorativa; con dati multi-anno diventa molto piu' solido.

## Report compatto da mandare per controllo

Esegui questa cella e manda l'output testuale completo. Contiene i numeri necessari per controllare se il trend e' interpretabile come degrado/domain shift o se e' probabilmente stagionalita'/artefatto.

In [ ]:
def _compact_report(out_dir: Path, top_n: int = 20):
    with open(out_dir / 'summary.json', encoding='utf-8') as f:
        summary = json.load(f)

    fleet_trends = pd.read_csv(out_dir / 'fleet_trend_summary.csv')
    plant_trends = pd.read_csv(out_dir / 'plant_trend_summary.csv')
    fleet_monthly = pd.read_csv(out_dir / 'fleet_monthly_performance.csv')
    plant_monthly = pd.read_csv(out_dir / 'plant_monthly_performance.csv')
    excluded_path = out_dir / 'excluded_implausible_pr.csv'
    excluded = pd.read_csv(excluded_path) if excluded_path.exists() else pd.DataFrame()
    diag_path = out_dir / 'excluded_implausible_pr_diagnostics.csv'
    excluded_diag = pd.read_csv(diag_path) if diag_path.exists() else pd.DataFrame()

    pr = plant_trends[plant_trends['metric'] == 'pr_pvgis_monthly'].copy()
    z = plant_trends[plant_trends['metric'] == 'pr_month_z_monthly'].copy()
    pr_top = pr.sort_values('slope_per_year').head(top_n)
    z_top = z.sort_values('slope_per_year').head(top_n)

    cols = [
        'plant', 'plant_id', 'n_points', 'mean_value', 'first_value', 'last_value',
        'slope_per_year', 'relative_change_pct_per_year', 'p_value',
        'kendall_tau', 'kendall_p_value', 'decreasing'
    ]

    print('=== SUMMARY.JSON ===')
    print(json.dumps(summary, indent=2))

    print('\n=== FLEET TREND SUMMARY ===')
    print(fleet_trends.to_string(index=False))

    print('\n=== FLEET MONTHLY PERFORMANCE ===')
    fm_cols = ['date', 'n_plants', 'actual_kwh', 'pvgis_expected_kwh', 'weighted_pr_pvgis', 'median_pr_pvgis', 'mean_pr_month_z']
    print(fleet_monthly[fm_cols].to_string(index=False))

    print(f'\n=== TOP {top_n} DECREASING PLANTS: PR_PVGIS ===')
    print(pr_top[cols].to_string(index=False))

    print(f'\n=== TOP {top_n} DECREASING PLANTS: MONTH-Z ===')
    print(z_top[cols].to_string(index=False))

    print('\n=== DATA COVERAGE / SOURCES ===')
    print('plant_monthly rows:', len(plant_monthly))
    print('plants in monthly table:', plant_monthly['plant'].nunique())
    print('kwp_source counts:')
    print(plant_monthly[['plant', 'kwp_source']].drop_duplicates()['kwp_source'].value_counts().to_string())
    print('valid monthly points per plant:')
    print(plant_monthly.groupby('plant')['pr_pvgis'].count().describe().to_string())
    print('\n=== EXCLUDED IMPLAUSIBLE PR PLANTS ===')
    if excluded.empty:
        print('none')
    else:
        ex_cols = ['plant', 'plant_id', 'mean_pr_pvgis', 'median_pr_pvgis', 'min_pr_pvgis', 'max_pr_pvgis', 'n_valid_months', 'kwp_used', 'kwp_source']
        print(excluded.sort_values('mean_pr_pvgis', ascending=False)[ex_cols].head(40).to_string(index=False))

    print('\n=== WHY PR IS IMPLAUSIBLE: COMPONENT DIAGNOSTICS ===')
    if excluded_diag.empty:
        print('none')
    else:
        diag_cols = [
            'plant', 'plant_id', 'mean_pr_pvgis', 'kwp_used', 'actual_sum_kwh',
            'expected_sum_kwh_pr1', 'actual_over_expected_sum', 'energy_p99',
            'energy_p99_over_kwp', 'energy_max', 'energy_max_over_kwp',
            'poa_kwm2_p99', 'poa_kwm2_max', 'diagnostic_flags'
        ]
        existing = [c for c in diag_cols if c in excluded_diag.columns]
        print(excluded_diag[existing].head(40).to_string(index=False))

_compact_report(OUT, top_n=20)